In [8]:
# Fit SARIMA with weekly seasonality (m=7)
sarima_model = fit_sarima_model(train_occupancy, seasonal_period=7)
print(sarima_model.summary())

Performing stepwise search to minimize aic
 ARIMA(2,0,2)(1,0,1)[7] intercept   : AIC=-341.805, Time=0.56 sec
 ARIMA(0,0,0)(0,0,0)[7] intercept   : AIC=-185.765, Time=0.02 sec
 ARIMA(1,0,0)(1,0,0)[7] intercept   : AIC=-319.324, Time=0.18 sec
 ARIMA(0,0,1)(0,0,1)[7] intercept   : AIC=-282.194, Time=0.10 sec
 ARIMA(0,0,0)(0,0,0)[7]             : AIC=197.251, Time=0.02 sec
 ARIMA(2,0,2)(0,0,1)[7] intercept   : AIC=-324.121, Time=0.32 sec
 ARIMA(2,0,2)(1,0,0)[7] intercept   : AIC=-324.198, Time=0.50 sec
 ARIMA(2,0,2)(2,0,1)[7] intercept   : AIC=-339.770, Time=0.83 sec
 ARIMA(2,0,2)(1,0,2)[7] intercept   : AIC=-345.305, Time=0.84 sec
 ARIMA(2,0,2)(0,0,2)[7] intercept   : AIC=-324.179, Time=0.57 sec
 ARIMA(2,0,2)(2,0,2)[7] intercept   : AIC=inf, Time=1.04 sec
 ARIMA(1,0,2)(1,0,2)[7] intercept   : AIC=-345.223, Time=0.70 sec
 ARIMA(2,0,1)(1,0,2)[7] intercept   : AIC=-344.605, Time=0.66 sec
 ARIMA(3,0,2)(1,0,2)[7] intercept   : AIC=-329.290, Time=1.00 sec
 ARIMA(2,0,3)(1,0,2)[7] intercept   : A

## Evaluation Metrics

### Demand Forecasting (SARIMA)

In [15]:
# ── SARIMA evaluation over the full test period ──────────────────────────────
occ_obs  = test_occupancy.values
occ_pred = forecasted_occupancy

mae_sarima  = np.mean(np.abs(occ_obs - occ_pred))
rmse_sarima = np.sqrt(np.mean((occ_obs - occ_pred) ** 2))
bias_sarima = np.mean(occ_pred - occ_obs)          # positive = model over-predicts

# Seasonal naive baseline: forecast for day h = observed value from same weekday
# in the last full week of training (h=0 → train[-7], h=7 → train[-7], etc.)
s = 7
naive_pred = np.array([train_occupancy.iloc[-s + (h % s)] for h in range(PREDICTION_HORIZON_DAYS)])
mae_naive  = np.mean(np.abs(occ_obs - naive_pred))

print("SARIMA forecast evaluation")
print(f"  Test period : {test_occupancy.index[0].date()} → {test_occupancy.index[-1].date()}  ({PREDICTION_HORIZON_DAYS} days)")
print(f"  MAE         : {mae_sarima:.4f}  ({mae_sarima*100:.2f} pp)")
print(f"  RMSE        : {rmse_sarima:.4f}  ({rmse_sarima*100:.2f} pp)")
print(f"  Mean bias   : {bias_sarima:+.4f}  ({bias_sarima*100:+.2f} pp)  [+= over-predicts, -= under-predicts]")
print()
print(f"  Naive seasonal baseline MAE : {mae_naive:.4f}  ({mae_naive*100:.2f} pp)")
print(f"  SARIMA improvement over naive : {(mae_naive - mae_sarima)*100:.2f} pp  ({(1 - mae_sarima/mae_naive)*100:.1f}% reduction)")

SARIMA forecast evaluation
  Test period : 2025-04-27 → 2025-05-26  (30 days)
  MAE         : 0.0784  (7.84 pp)
  RMSE        : 0.0977  (9.77 pp)
  Mean bias   : -0.0150  (-1.50 pp)  [+= over-predicts, -= under-predicts]

  Naive seasonal baseline MAE : 0.0936  (9.36 pp)
  SARIMA improvement over naive : 1.51 pp  (16.2% reduction)


### Pricing Recommendations

In [16]:
# ── Observed prices per apartment per day in the test period ─────────────────
# Weighted average nightly_bed_rate (weight = beds_count) to match how base prices
# are defined, so the relative errors are on a consistent scale.
test_daily = daily_stays[daily_stays['stay_date'].isin(forecasted_dates)].copy()

test_daily['weighted_price'] = test_daily['nightly_bed_rate'] * test_daily['beds_count']

obs_prices = (
    test_daily
    .groupby(['stay_date', 'apartment_id'])
    .agg(weighted_price_sum=('weighted_price', 'sum'), beds_sum=('beds_count', 'sum'))
    .reset_index()
)
obs_prices['obs_price_per_bed'] = obs_prices['weighted_price_sum'] / obs_prices['beds_sum']
obs_prices = obs_prices.drop(columns=['weighted_price_sum', 'beds_sum'])
obs_prices['stay_date'] = pd.to_datetime(obs_prices['stay_date'])

# ── Recommended prices in long format ────────────────────────────────────────
apt_cols = [c for c in recommendations.columns if c.startswith('apt_') and c.endswith('_price')]
rec_long = (
    recommendations[apt_cols]
    .reset_index()
    .melt(id_vars='date', var_name='apt_col', value_name='rec_price_per_bed')
)
rec_long['apartment_id'] = rec_long['apt_col'].str.extract(r'apt_(\d+)_price').astype(int)
rec_long = rec_long.drop(columns='apt_col')

# ── Merge on date × apartment ─────────────────────────────────────────────────
eval_df = rec_long.merge(
    obs_prices,
    left_on=['date', 'apartment_id'],
    right_on=['stay_date', 'apartment_id'],
    how='inner',
).drop(columns='stay_date')

# Relative price error: (recommended - observed) / observed
eval_df['rel_error'] = (eval_df['rec_price_per_bed'] - eval_df['obs_price_per_bed']) / eval_df['obs_price_per_bed']

# Relative price of each apartment on each day (observed vs its own base price)
eval_df['obs_relative'] = eval_df['apartment_id'].map(base_prices)
eval_df['obs_relative'] = eval_df['obs_price_per_bed'] / eval_df['obs_relative']
eval_df['rec_relative'] = eval_df['rec_price_per_bed'] / eval_df['apartment_id'].map(base_prices)

print(f"Evaluation pairs: {len(eval_df)}  "
      f"({eval_df['apartment_id'].nunique()} apartments × test dates with at least one booking)")

Evaluation pairs: 290  (12 apartments × test dates with at least one booking)


In [17]:
# ── Aggregate pricing evaluation metrics ─────────────────────────────────────
mean_bias   = eval_df['rel_error'].mean()          # signed: negative = under-pricing
mae_price   = eval_df['rel_error'].abs().mean()
std_price   = eval_df['rel_error'].std()

# Directional accuracy: does the model correctly predict whether to price
# above or below each apartment's historical average (base price)?
# A tie (either side exactly at 1.0) is excluded as ambiguous.
dir_df = eval_df[(eval_df['obs_relative'] != 1.0) & (eval_df['rec_relative'] != 1.0)].copy()
dir_df['direction_correct'] = np.sign(dir_df['rec_relative'] - 1.0) == np.sign(dir_df['obs_relative'] - 1.0)
directional_accuracy = dir_df['direction_correct'].mean()

print("Pricing recommendation evaluation  (α = {:.1f}, full test period, all apartments)".format(PRICING_QUANTILE))
print(f"  Evaluation pairs         : {len(eval_df)}")
print(f"  Mean signed error (bias) : {mean_bias*100:+.1f}%  [negative = model under-prices]")
print(f"  Mean absolute error      : {mae_price*100:.1f}%")
print(f"  Std dev of error         : {std_price*100:.1f}%")
print(f"  Directional accuracy     : {directional_accuracy*100:.1f}%  ({dir_df['direction_correct'].sum()}/{len(dir_df)} pairs)")

Pricing recommendation evaluation  (α = 0.5, full test period, all apartments)
  Evaluation pairs         : 290
  Mean signed error (bias) : -13.4%  [negative = model under-prices]
  Mean absolute error      : 29.0%
  Std dev of error         : 31.8%
  Directional accuracy     : 41.7%  (121/290 pairs)


In [18]:
# ── Per-apartment breakdown ───────────────────────────────────────────────────
apt_summary = (
    eval_df
    .groupby('apartment_id')['rel_error']
    .agg(
        n='count',
        mean_bias=lambda x: x.mean(),
        mae=lambda x: x.abs().mean(),
        std='std',
    )
    .reset_index()
)
apt_summary[['mean_bias', 'mae', 'std']] *= 100   # convert to %

dir_per_apt = (
    dir_df.groupby('apartment_id')['direction_correct']
    .mean()
    .mul(100)
    .rename('dir_acc_%')
    .reset_index()
)
apt_summary = apt_summary.merge(dir_per_apt, on='apartment_id', how='left')

apt_summary = apt_summary.rename(columns={
    'n':         'pairs',
    'mean_bias': 'mean_bias_%',
    'mae':       'mae_%',
    'std':       'std_%',
})

print("Per-apartment pricing evaluation:")
print(apt_summary.round(1).to_string(index=False))

Per-apartment pricing evaluation:
 apartment_id  pairs  mean_bias_%  mae_%  std_%  dir_acc_%
            0     25        -25.8   39.1   38.1       32.0
            1     26        -13.9   23.7   23.2       34.6
            2     26          7.2   29.1   37.5       88.5
            3     25        -12.1   20.6   22.6       40.0
            4     23        -36.7   37.0   22.1       26.1
            5     23         -8.2   18.5   19.9       43.5
            6     20          5.2   30.6   40.1       65.0
            7     24        -27.3   41.6   33.1       16.7
            8     29        -20.3   24.2   19.4       27.6
            9     26        -15.9   32.1   31.7       30.8
           11     19         -7.8   16.4   18.7       52.6
           12     24         -1.7   33.9   38.4       50.0
